In [0]:
!pip install spotipy

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
from pyspark.sql import SparkSession

In [0]:
#Creamos las session de apache spark en una variable

spark = SparkSession.builder.getOrCreate()

In [0]:
# spark.stop()

In [0]:
# import spotify packages

import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

In [0]:
# create a proyect and retrieval the credentials 

# https://developer.spotify.com/documentation/web-api

client_id = "42e39ebda6fc49d48d9a3b3548317b88"  # your own client id
client_secret = "2b200bcd78284aa4b6e2c8a041e536e9" # your own client secret

In [0]:
# Create a Spotify client

client_credentials_manager = SpotifyClientCredentials(client_id=client_id, client_secret=client_secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

In [0]:
# retrieval your playlist

top_tracks = sp.playlist_tracks('7EbiTIskJft6zwA8WDC4pb', limit=100) #7EbiTIskJft6zwA8WDC4pb / 37i9dQZF1DXbITWG1ZJKYt / 3VzB0idVneaUskLFv8j47y

In [0]:
import pandas as pd
import seaborn as sns

import warnings 
warnings.filterwarnings('ignore')

In [0]:
# extract from the playlist 

list_all_song = []

for track in top_tracks['items']:

    track_name = track['track']['name']
    artist_name = track['track']['artists'][0]['name']
    popularity = track['track']['popularity']
    explicit = track['track']['explicit']

    release_date = track['track']['album']['release_date']
    duration_min = track['track']['duration_ms']/60000 # convert ms to min

    list_all_song.append({'track_name':track_name, 'artist_name':artist_name, 'popularity':popularity, 'explicit': explicit,'release_date':release_date,'duration_min':duration_min})

df_all_songs = pd.DataFrame(list_all_song)

In [0]:
df_all_songs.head()

,track_name,artist_name,popularity,explicit,release_date,duration_min
0,Rompelo Ahí,Los Nota Lokos,52,False,2014-04-16,2.608367
1,Toy de gira,Corona,31,True,2021-06-03,3.382850
2,Olha A Explosão - Remix,MC Kevinho,47,False,2018-04-27,3.512883
3,Vidrado Em Você,Dj Guuga,79,False,2019-06-11,2.246150
4,Rabetão,MC Lan,61,True,2017-03-10,2.844050


In [0]:
df_all_songs.shape

(50, 6)

In [0]:
# cast date

df_all_songs['release_date'] = pd.to_datetime(df_all_songs['release_date'], errors='ignore')

In [0]:
def get_bracket_date(x):
    
    if x <= '2000-01-01':
        return 'Antes de los 2000'
    elif x > '2000-01-01' and x <= '2010-01-01':
        return 'Entre 2000 y 2010'
    elif x > '2010-01-01' and x <= '2020-01-01':
        return 'Entre 2010 y 2020'
    else:
        return 'Después de 2020'

In [0]:
df_all_songs['release_date_bracket'] = df_all_songs['release_date'].apply(get_bracket_date)

In [0]:
df_all_songs.head()

,track_name,artist_name,popularity,explicit,release_date,duration_min,release_date_bracket
0,Rompelo Ahí,Los Nota Lokos,52,False,2014-04-16,2.608367,Entre 2010 y 2020
1,Toy de gira,Corona,31,True,2021-06-03,3.382850,Después de 2020
2,Olha A Explosão - Remix,MC Kevinho,47,False,2018-04-27,3.512883,Entre 2010 y 2020
3,Vidrado Em Você,Dj Guuga,79,False,2019-06-11,2.246150,Entre 2010 y 2020
4,Rabetão,MC Lan,61,True,2017-03-10,2.844050,Entre 2010 y 2020


# Introduction to Spark

In [0]:
# create a spark dataframe

df_spark = spark.createDataFrame(df_all_songs)

In [0]:
df_spark

DataFrame[track_name: string, artist_name: string, popularity: bigint, explicit: boolean, release_date: string, duration_min: double, release_date_bracket: string]

In [0]:
# print data type

type(df_spark)

pyspark.sql.connect.dataframe.DataFrame

In [0]:
df_spark.show()

+--------------------+----------------+----------+--------+------------+------------------+--------------------+
|          track_name|     artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|
+--------------------+----------------+----------+--------+------------+------------------+--------------------+
|         Rompelo Ahí|  Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020|
|         Toy de gira|          Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020|
|Olha A Explosão -...|      MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|
|     Vidrado Em Você|        Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|
|             Rabetão|          MC Lan|        61|    true|  2017-03-10|           2.84405|   Entre 2010 y 2020|
|Ele Te Bota Soca ...|       MC Mazzie|        61|    true|  2021-06-18|           3.01545|     

In [0]:
df_spark.show(5)

+--------------------+--------------+----------+--------+------------+------------------+--------------------+
|          track_name|   artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+
|         Rompelo Ahí|Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020|
|         Toy de gira|        Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020|
|Olha A Explosão -...|    MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|
|     Vidrado Em Você|      Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|
|             Rabetão|        MC Lan|        61|    true|  2017-03-10|           2.84405|   Entre 2010 y 2020|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+
o

In [0]:
df_spark.printSchema()

root
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- popularity: long (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- release_date: string (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- release_date_bracket: string (nullable = true)



In [0]:
# select a subset of columns

df_spark.select('track_name','explicit').show()

+--------------------+--------+
|          track_name|explicit|
+--------------------+--------+
|         Rompelo Ahí|   false|
|         Toy de gira|    true|
|Olha A Explosão -...|   false|
|     Vidrado Em Você|   false|
|             Rabetão|    true|
|Ele Te Bota Soca ...|    true|
|     Bum Bum Tam Tam|    true|
|     MONTAGEM XONADA|   false|
|           Galáctico|    true|
|    Parado no Bailão|   false|
|            Butakera|   false|
|Scarface (Push It...|   false|
|      Tubarão Te Amo|    true|
|             Vem Vem|    true|
|         DIA DELÍCIA|    true|
|        VEM NO PIQUE|   false|
|  LUNA BALA - Slowed|   false|
|     MONTAGEM TOMADA|   false|
|         ESTELA LUNA|   false|
|     MONTAGEM BAILÃO|    true|
+--------------------+--------+
only showing top 20 rows


In [0]:
from pyspark.sql.functions import col

In [0]:
df_spark.select(col("track_name"),col("explicit")).show()

+--------------------+--------+
|          track_name|explicit|
+--------------------+--------+
|         Rompelo Ahí|   false|
|         Toy de gira|    true|
|Olha A Explosão -...|   false|
|     Vidrado Em Você|   false|
|             Rabetão|    true|
|Ele Te Bota Soca ...|    true|
|     Bum Bum Tam Tam|    true|
|     MONTAGEM XONADA|   false|
|           Galáctico|    true|
|    Parado no Bailão|   false|
|            Butakera|   false|
|Scarface (Push It...|   false|
|      Tubarão Te Amo|    true|
|             Vem Vem|    true|
|         DIA DELÍCIA|    true|
|        VEM NO PIQUE|   false|
|  LUNA BALA - Slowed|   false|
|     MONTAGEM TOMADA|   false|
|         ESTELA LUNA|   false|
|     MONTAGEM BAILÃO|    true|
+--------------------+--------+
only showing top 20 rows


In [0]:
df_spark.select('track_name','explicit').show()

+--------------------+--------+
|          track_name|explicit|
+--------------------+--------+
|         Rompelo Ahí|   false|
|         Toy de gira|    true|
|Olha A Explosão -...|   false|
|     Vidrado Em Você|   false|
|             Rabetão|    true|
|Ele Te Bota Soca ...|    true|
|     Bum Bum Tam Tam|    true|
|     MONTAGEM XONADA|   false|
|           Galáctico|    true|
|    Parado no Bailão|   false|
|            Butakera|   false|
|Scarface (Push It...|   false|
|      Tubarão Te Amo|    true|
|             Vem Vem|    true|
|         DIA DELÍCIA|    true|
|        VEM NO PIQUE|   false|
|  LUNA BALA - Slowed|   false|
|     MONTAGEM TOMADA|   false|
|         ESTELA LUNA|   false|
|     MONTAGEM BAILÃO|    true|
+--------------------+--------+
only showing top 20 rows


In [0]:
df_spark.select(df_spark.colRegex("`^.*name*`")).show()

+--------------------+----------------+
|          track_name|     artist_name|
+--------------------+----------------+
|         Rompelo Ahí|  Los Nota Lokos|
|         Toy de gira|          Corona|
|Olha A Explosão -...|      MC Kevinho|
|     Vidrado Em Você|        Dj Guuga|
|             Rabetão|          MC Lan|
|Ele Te Bota Soca ...|       MC Mazzie|
|     Bum Bum Tam Tam|        MC Fioti|
|     MONTAGEM XONADA|            MXZI|
|           Galáctico|     Young Jairo|
|    Parado no Bailão|   MC L da Vinte|
|            Butakera|       La Joaqui|
|Scarface (Push It...|   Paul Engemann|
|      Tubarão Te Amo|Dj LK da Escócia|
|             Vem Vem|         Jmilton|
|         DIA DELÍCIA|          Nakama|
|        VEM NO PIQUE|             MGD|
|  LUNA BALA - Slowed|     Yb Wasg'ood|
|     MONTAGEM TOMADA|            MXZI|
|         ESTELA LUNA|        OHMAN.PH|
|     MONTAGEM BAILÃO|          Repsaj|
+--------------------+----------------+
only showing top 20 rows


In [0]:
df_spark.show(5)

+--------------------+--------------+----------+--------+------------+------------------+--------------------+
|          track_name|   artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+
|         Rompelo Ahí|Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020|
|         Toy de gira|        Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020|
|Olha A Explosão -...|    MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|
|     Vidrado Em Você|      Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|
|             Rabetão|        MC Lan|        61|    true|  2017-03-10|           2.84405|   Entre 2010 y 2020|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+
o

In [0]:
# cast column

df_spark.withColumn("popularity",col("popularity").cast("Integer")).show()

+--------------------+----------------+----------+--------+------------+------------------+--------------------+
|          track_name|     artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|
+--------------------+----------------+----------+--------+------------+------------------+--------------------+
|         Rompelo Ahí|  Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020|
|         Toy de gira|          Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020|
|Olha A Explosão -...|      MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|
|     Vidrado Em Você|        Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|
|             Rabetão|          MC Lan|        61|    true|  2017-03-10|           2.84405|   Entre 2010 y 2020|
|Ele Te Bota Soca ...|       MC Mazzie|        61|    true|  2021-06-18|           3.01545|     

In [0]:
df_spark.printSchema()

root
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- popularity: long (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- release_date: string (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- release_date_bracket: string (nullable = true)



In [0]:
df_spark = df_spark.withColumn("popularity", col("popularity").cast("integer"))

In [0]:
df_spark.printSchema()

root
 |-- track_name: string (nullable = true)
 |-- artist_name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- release_date: string (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- release_date_bracket: string (nullable = true)



In [0]:
df_spark.withColumn("duration_hrs",col("duration_min")/60).show(5)

+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+
|          track_name|   artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+
|         Rompelo Ahí|Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020| 0.04347277777777778|
|         Toy de gira|        Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020| 0.05638083333333333|
|Olha A Explosão -...|    MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|0.058548055555555555|
|     Vidrado Em Você|      Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|0.037435833333333335|
|             Rabetão|        MC Lan|        61|    true|  2017-03-10|      

In [0]:
df_spark.show(5)

+--------------------+--------------+----------+--------+------------+------------------+--------------------+
|          track_name|   artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+
|         Rompelo Ahí|Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020|
|         Toy de gira|        Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020|
|Olha A Explosão -...|    MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|
|     Vidrado Em Você|      Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|
|             Rabetão|        MC Lan|        61|    true|  2017-03-10|           2.84405|   Entre 2010 y 2020|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+
o

In [0]:
df_spark = df_spark.withColumn("duration_hrs", col("duration_min") / 60)

In [0]:
df_spark.show(5)

+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+
|          track_name|   artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+
|         Rompelo Ahí|Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020| 0.04347277777777778|
|         Toy de gira|        Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020| 0.05638083333333333|
|Olha A Explosão -...|    MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|0.058548055555555555|
|     Vidrado Em Você|      Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|0.037435833333333335|
|             Rabetão|        MC Lan|        61|    true|  2017-03-10|      

In [0]:
df_spark = df_spark.withColumnRenamed("track_name","pista")

In [0]:
df_spark.show(5)

+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+
|               pista|   artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+
|         Rompelo Ahí|Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020| 0.04347277777777778|
|         Toy de gira|        Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020| 0.05638083333333333|
|Olha A Explosão -...|    MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|0.058548055555555555|
|     Vidrado Em Você|      Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|0.037435833333333335|
|             Rabetão|        MC Lan|        61|    true|  2017-03-10|      

In [0]:
from pyspark.sql.functions import lit

In [0]:
df_spark = df_spark.withColumn("user", lit("joao"))

In [0]:
df_spark.show(5)

+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|               pista|   artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+--------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|         Rompelo Ahí|Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020| 0.04347277777777778|joao|
|         Toy de gira|        Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020| 0.05638083333333333|joao|
|Olha A Explosão -...|    MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|0.058548055555555555|joao|
|     Vidrado Em Você|      Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|0.037435833333333335|joao|
|             Rabetão|        MC Lan|    

# Filter & Sort functions

In [0]:
df_spark.filter(df_spark.release_date_bracket == "Después de 2020").show(truncate=False)

+---------------------+----------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|pista                |artist_name     |popularity|explicit|release_date|duration_min      |release_date_bracket|duration_hrs        |user|
+---------------------+----------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|Toy de gira          |Corona          |31        |true    |2021-06-03  |3.38285           |Después de 2020     |0.05638083333333333 |joao|
|Ele Te Bota Soca Soca|MC Mazzie       |61        |true    |2021-06-18  |3.01545           |Después de 2020     |0.0502575           |joao|
|MONTAGEM XONADA      |MXZI            |87        |false   |2025-07-02  |1.2613666666666667|Después de 2020     |0.02102277777777778 |joao|
|Galáctico            |Young Jairo     |26        |true    |2024-09-06  |3.216483333333333 |Después de 2020     |0.053608055555555555|joao|
|Butakera           

In [0]:
df_spark.filter(df_spark.release_date_bracket == "Después de 2020").count()

25

In [0]:
df_spark.filter(df_spark.release_date_bracket != "Después de 2020").show(truncate=False) 

+-------------------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|pista                          |artist_name    |popularity|explicit|release_date|duration_min      |release_date_bracket|duration_hrs        |user|
+-------------------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|Rompelo Ahí                    |Los Nota Lokos |52        |false   |2014-04-16  |2.6083666666666665|Entre 2010 y 2020   |0.04347277777777778 |joao|
|Olha A Explosão - Remix        |MC Kevinho     |47        |false   |2018-04-27  |3.5128833333333334|Entre 2010 y 2020   |0.058548055555555555|joao|
|Vidrado Em Você                |Dj Guuga       |79        |false   |2019-06-11  |2.24615           |Entre 2010 y 2020   |0.037435833333333335|joao|
|Rabetão                        |MC Lan         |61        |true    |2017-03-10  |2.84405           |Entre

In [0]:
df_spark.filter(~(df_spark.release_date_bracket == "Después de 2020")).show(truncate=False)

+-------------------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|pista                          |artist_name    |popularity|explicit|release_date|duration_min      |release_date_bracket|duration_hrs        |user|
+-------------------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|Rompelo Ahí                    |Los Nota Lokos |52        |false   |2014-04-16  |2.6083666666666665|Entre 2010 y 2020   |0.04347277777777778 |joao|
|Olha A Explosão - Remix        |MC Kevinho     |47        |false   |2018-04-27  |3.5128833333333334|Entre 2010 y 2020   |0.058548055555555555|joao|
|Vidrado Em Você                |Dj Guuga       |79        |false   |2019-06-11  |2.24615           |Entre 2010 y 2020   |0.037435833333333335|joao|
|Rabetão                        |MC Lan         |61        |true    |2017-03-10  |2.84405           |Entre

In [0]:
df_spark.filter(col("artist_name") == "Los Nota Lokos").show(truncate=False) 

+----------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|pista                 |artist_name   |popularity|explicit|release_date|duration_min      |release_date_bracket|duration_hrs        |user|
+----------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|Rompelo Ahí           |Los Nota Lokos|52        |false   |2014-04-16  |2.6083666666666665|Entre 2010 y 2020   |0.04347277777777778 |joao|
|La Mas Linda del Salón|Los Nota Lokos|43        |false   |2014-09-08  |3.1368666666666667|Entre 2010 y 2020   |0.05228111111111111 |joao|
|Mi Nena Facebook      |Los Nota Lokos|56        |false   |2014-04-21  |2.8842            |Entre 2010 y 2020   |0.048069999999999995|joao|
+----------------------+--------------+----------+--------+------------+------------------+--------------------+--------------------+----+



In [0]:
df_spark.filter("explicit == False").show()

+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|               pista|    artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|         Rompelo Ahí| Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020| 0.04347277777777778|joao|
|Olha A Explosão -...|     MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|0.058548055555555555|joao|
|     Vidrado Em Você|       Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|0.037435833333333335|joao|
|     MONTAGEM XONADA|           MXZI|        87|   false|  2025-07-02|1.2613666666666667|     Después de 2020| 0.02102277777777778|joao|
|    Parado no Bailão|  MC L da Vi

In [0]:
df_spark.filter("duration_min > 3").show()

+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|               pista|    artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|         Toy de gira|         Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020| 0.05638083333333333|joao|
|Olha A Explosão -...|     MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|0.058548055555555555|joao|
|Ele Te Bota Soca ...|      MC Mazzie|        61|    true|  2021-06-18|           3.01545|     Después de 2020|           0.0502575|joao|
|     Bum Bum Tam Tam|       MC Fioti|        67|    true|  2017-12-15|3.5664666666666665|   Entre 2010 y 2020|0.059441111111111104|joao|
|           Galáctico|    Young Ja

In [0]:
df_spark.filter( (df_spark.popularity >80) & (df_spark.duration_min  >= 3) ).show(truncate=False)  

+----------+---------------+----------+--------+------------+------------------+--------------------+-------------------+----+
|pista     |artist_name    |popularity|explicit|release_date|duration_min      |release_date_bracket|duration_hrs       |user|
+----------+---------------+----------+--------+------------+------------------+--------------------+-------------------+----+
|Pump It   |Black Eyed Peas|82        |true    |2005-01-01  |3.5511            |Entre 2000 y 2010   |0.059185           |joao|
|Poker Face|Lady Gaga      |85        |true    |2008-01-01  |3.953333333333333 |Entre 2000 y 2010   |0.06588888888888889|joao|
|No Lie    |Sean Paul      |85        |false   |2018-06-29  |3.6862666666666666|Entre 2010 y 2020   |0.06143777777777778|joao|
+----------+---------------+----------+--------+------------+------------------+--------------------+-------------------+----+



In [0]:
list_date_bckt = ["Después de 2020","Entre 2000 y 2010"]

df_spark.filter(df_spark.release_date_bracket.isin(list_date_bckt)).show()

+--------------------+----------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|               pista|     artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+--------------------+----------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|         Toy de gira|          Corona|        31|    true|  2021-06-03|           3.38285|     Después de 2020| 0.05638083333333333|joao|
|Ele Te Bota Soca ...|       MC Mazzie|        61|    true|  2021-06-18|           3.01545|     Después de 2020|           0.0502575|joao|
|     MONTAGEM XONADA|            MXZI|        87|   false|  2025-07-02|1.2613666666666667|     Después de 2020| 0.02102277777777778|joao|
|           Galáctico|     Young Jairo|        26|    true|  2024-09-06| 3.216483333333333|     Después de 2020|0.053608055555555555|joao|
|            Butakera|     

In [0]:
df_spark.filter(~df_spark.release_date_bracket.isin(list_date_bckt)).show()

+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|               pista|    artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|         Rompelo Ahí| Los Nota Lokos|        52|   false|  2014-04-16|2.6083666666666665|   Entre 2010 y 2020| 0.04347277777777778|joao|
|Olha A Explosão -...|     MC Kevinho|        47|   false|  2018-04-27|3.5128833333333334|   Entre 2010 y 2020|0.058548055555555555|joao|
|     Vidrado Em Você|       Dj Guuga|        79|   false|  2019-06-11|           2.24615|   Entre 2010 y 2020|0.037435833333333335|joao|
|             Rabetão|         MC Lan|        61|    true|  2017-03-10|           2.84405|   Entre 2010 y 2020|0.047400833333333336|joao|
|     Bum Bum Tam Tam|       MC Fi

In [0]:
df_spark.filter(df_spark.artist_name.like("%aga%")).show()

+----------+-----------+----------+--------+------------+-----------------+--------------------+-------------------+----+
|     pista|artist_name|popularity|explicit|release_date|     duration_min|release_date_bracket|       duration_hrs|user|
+----------+-----------+----------+--------+------------+-----------------+--------------------+-------------------+----+
|Poker Face|  Lady Gaga|        85|    true|  2008-01-01|3.953333333333333|   Entre 2000 y 2010|0.06588888888888889|joao|
+----------+-----------+----------+--------+------------+-----------------+--------------------+-------------------+----+



In [0]:
df_spark.count()

50

In [0]:
df_spark_distinct = df_spark.distinct()
df_spark_distinct.count(), df_spark.count() # no hay repetidos

(50, 50)

In [0]:
df_spark_distinct = df_spark.dropDuplicates()
df_spark_distinct.count(),  df_spark.count()

(50, 50)

In [0]:
df_spark.sort('popularity', ascending = False).show(10)

+------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|             pista|    artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|   MONTAGEM XONADA|           MXZI|        87|   false|  2025-07-02|1.2613666666666667|     Después de 2020| 0.02102277777777778|joao|
|        Poker Face|      Lady Gaga|        85|    true|  2008-01-01| 3.953333333333333|   Entre 2000 y 2010| 0.06588888888888889|joao|
|            No Lie|      Sean Paul|        85|   false|  2018-06-29|3.6862666666666666|   Entre 2010 y 2020| 0.06143777777777778|joao|
|           Pump It|Black Eyed Peas|        82|    true|  2005-01-01|            3.5511|   Entre 2000 y 2010|            0.059185|joao|
|   Me Porto Bonito|      Bad Bunny|        82| 

In [0]:
df_spark.sort('popularity','duration_min', ascending = False).show(10)

+------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|             pista|    artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|   MONTAGEM XONADA|           MXZI|        87|   false|  2025-07-02|1.2613666666666667|     Después de 2020| 0.02102277777777778|joao|
|        Poker Face|      Lady Gaga|        85|    true|  2008-01-01| 3.953333333333333|   Entre 2000 y 2010| 0.06588888888888889|joao|
|            No Lie|      Sean Paul|        85|   false|  2018-06-29|3.6862666666666666|   Entre 2010 y 2020| 0.06143777777777778|joao|
|           Pump It|Black Eyed Peas|        82|    true|  2005-01-01|            3.5511|   Entre 2000 y 2010|            0.059185|joao|
|   Me Porto Bonito|      Bad Bunny|        82| 

In [0]:
df_spark.orderBy("popularity","duration_min", ascending = False).show(10)

+------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|             pista|    artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|   MONTAGEM XONADA|           MXZI|        87|   false|  2025-07-02|1.2613666666666667|     Después de 2020| 0.02102277777777778|joao|
|        Poker Face|      Lady Gaga|        85|    true|  2008-01-01| 3.953333333333333|   Entre 2000 y 2010| 0.06588888888888889|joao|
|            No Lie|      Sean Paul|        85|   false|  2018-06-29|3.6862666666666666|   Entre 2010 y 2020| 0.06143777777777778|joao|
|           Pump It|Black Eyed Peas|        82|    true|  2005-01-01|            3.5511|   Entre 2000 y 2010|            0.059185|joao|
|   Me Porto Bonito|      Bad Bunny|        82| 

In [0]:
df_spark.orderBy("duration_min", ascending = False).show(10)

+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|               pista|    artist_name|popularity|explicit|release_date|      duration_min|release_date_bracket|        duration_hrs|user|
+--------------------+---------------+----------+--------+------------+------------------+--------------------+--------------------+----+
|The Time (Dirty Bit)|Black Eyed Peas|        73|   false|  2010-01-01|5.1273333333333335|   Entre 2000 y 2010| 0.08545555555555556|joao|
|                Fiel|Los Legendarios|        65|   false|  2021-02-04| 4.361083333333333|     Después de 2020| 0.07268472222222222|joao|
|           Que Calor|     Supermerk2|        56|   false|  2003-09-23|           4.07755|   Entre 2000 y 2010| 0.06795916666666665|joao|
|          Poker Face|      Lady Gaga|        85|    true|  2008-01-01| 3.953333333333333|   Entre 2000 y 2010| 0.06588888888888889|joao|
|              No Lie|      Sean P

In [0]:
df_spark.createOrReplaceTempView("joao_playlist")

In [0]:
spark.sql("select pista,artist_name, popularity from joao_playlist ORDER BY popularity desc").show(10)

+------------------+---------------+----------+
|             pista|    artist_name|popularity|
+------------------+---------------+----------+
|   MONTAGEM XONADA|           MXZI|        87|
|        Poker Face|      Lady Gaga|        85|
|            No Lie|      Sean Paul|        85|
|           Pump It|Black Eyed Peas|        82|
|   Me Porto Bonito|      Bad Bunny|        82|
|LUNA BALA - Slowed|    Yb Wasg'ood|        82|
|   MONTAGEM BAILÃO|         Repsaj|        81|
|           Vem Vem|        Jmilton|        81|
|   Vidrado Em Você|       Dj Guuga|        79|
|             Rompe|   Daddy Yankee|        78|
+------------------+---------------+----------+
only showing top 10 rows


In [0]:
%sql
select pista,artist_name, popularity from joao_playlist ORDER BY popularity desc

pista,artist_name,popularity
MONTAGEM XONADA,MXZI,87
Poker Face,Lady Gaga,85
No Lie,Sean Paul,85
LUNA BALA - Slowed,Yb Wasg'ood,82
Pump It,Black Eyed Peas,82
Me Porto Bonito,Bad Bunny,82
Vem Vem,Jmilton,81
MONTAGEM BAILÃO,Repsaj,81
Vidrado Em Você,Dj Guuga,79
Rompe,Daddy Yankee,78


In [0]:
df_result = spark.sql("""
    SELECT pista, artist_name, popularity 
    FROM joao_playlist
    ORDER BY popularity DESC
""")

In [0]:
df_result.show(5)

+------------------+---------------+----------+
|             pista|    artist_name|popularity|
+------------------+---------------+----------+
|   MONTAGEM XONADA|           MXZI|        87|
|            No Lie|      Sean Paul|        85|
|        Poker Face|      Lady Gaga|        85|
|           Pump It|Black Eyed Peas|        82|
|LUNA BALA - Slowed|    Yb Wasg'ood|        82|
+------------------+---------------+----------+
only showing top 5 rows


In [0]:
df_result.count()

50

In [0]:
%sql
select pista,artist_name, popularity from joao_playlist ORDER By popularity desc

pista,artist_name,popularity
MONTAGEM XONADA,MXZI,87
Poker Face,Lady Gaga,85
No Lie,Sean Paul,85
LUNA BALA - Slowed,Yb Wasg'ood,82
Pump It,Black Eyed Peas,82
Me Porto Bonito,Bad Bunny,82
Vem Vem,Jmilton,81
MONTAGEM BAILÃO,Repsaj,81
Vidrado Em Você,Dj Guuga,79
Rompe,Daddy Yankee,78


# Agg

In [0]:
from pyspark.sql.functions import col,sum,avg,max,min,countDistinct, median

In [0]:
# check some metrics

# df_all_songs.groupby('artist_name').agg({'track_name':'nunique','popularity':'mean','duration_min':'mean','release_date':'max'}).sort_values('track_name', ascending = False).head(10)

In [0]:
df_spark.groupBy("artist_name").agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         avg("duration_min").alias("avg_duration_min"), \
         max("release_date").alias("max_release_date")
     ) \
    .sort('nunique_pista', ascending = False).show(20)

+----------------+-------------+------------------+------------------+----------------+
|     artist_name|nunique_pista|    avg_popularity|  avg_duration_min|max_release_date|
+----------------+-------------+------------------+------------------+----------------+
|  Los Nota Lokos|            3|50.333333333333336|2.8764777777777777|      2014-09-08|
|          DENNIS|            2|              62.0|2.9966583333333334|      2023-08-03|
|       Bad Bunny|            2|              78.0|           2.92245|      2022-05-06|
|            MXZI|            2|              48.5|1.2256166666666668|      2025-07-02|
| Black Eyed Peas|            2|              77.5| 4.339216666666667|      2010-01-01|
|      MC Kevinho|            2|              60.5| 3.106883333333333|      2018-04-27|
|      Supermerk2|            2|              57.5| 3.823108333333333|      2003-09-23|
|Dj LK da Escócia|            1|              62.0|            2.4846|      2022-09-28|
|       La Joaqui|            1|

In [0]:
df_spark.groupBy('artist_name').avg('popularity').sort('avg(popularity)', ascending = False).show(10)

+---------------+---------------+
|    artist_name|avg(popularity)|
+---------------+---------------+
|      Lady Gaga|           85.0|
|      Sean Paul|           85.0|
|    Yb Wasg'ood|           82.0|
|        Jmilton|           81.0|
|         Repsaj|           81.0|
|       Dj Guuga|           79.0|
|      Bad Bunny|           78.0|
|   Daddy Yankee|           78.0|
|Black Eyed Peas|           77.5|
|DJ Biel Divulga|           77.0|
+---------------+---------------+
only showing top 10 rows


In [0]:
# df_all_songs.groupby('release_date_bracket').agg({'track_name':'nunique','popularity':'mean','duration_min':'mean'}).sort_values('popularity', ascending = False)

In [0]:
df_spark.groupBy('release_date_bracket').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         avg("duration_min").alias("avg_duration_min")
     ) \
    .sort('avg_popularity', ascending = False).show(20)

+--------------------+-------------+-----------------+------------------+
|release_date_bracket|nunique_pista|   avg_popularity|  avg_duration_min|
+--------------------+-------------+-----------------+------------------+
|   Entre 2000 y 2010|            7|70.28571428571429|3.7502166666666668|
|   Antes de los 2000|            1|             68.0| 2.974666666666667|
|   Entre 2010 y 2020|           17|59.94117647058823| 3.033630392156863|
|     Después de 2020|           25|             58.6|2.4004866666666667|
+--------------------+-------------+-----------------+------------------+



In [0]:
# df_all_songs.groupby('explicit').agg({'popularity':['mean','median'],'track_name':'nunique','duration_min':['mean','median']})

In [0]:
df_spark.groupBy('explicit').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         median("popularity").alias("median_popularity"), \
         avg("duration_min").alias("avg_duration_min"),\
        median("duration_min").alias("median_duration_min")
     ).show(20)

+--------+-------------+-----------------+-----------------+-----------------+-------------------+
|explicit|nunique_pista|   avg_popularity|median_popularity| avg_duration_min|median_duration_min|
+--------+-------------+-----------------+-----------------+-----------------+-------------------+
|   false|           33|59.27272727272727|             59.0|2.829083838383839| 2.8297666666666665|
|    true|           17|             64.0|             70.0|2.791194117647059| 2.9761166666666665|
+--------+-------------+-----------------+-----------------+-----------------+-------------------+



In [0]:
df_spark.groupBy('explicit').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         median("popularity").alias("median_popularity"), \
         avg("duration_min").alias("avg_duration_min"),\
        median("duration_min").alias("median_duration_min")
     ).where(col("median_popularity") >= 60).show(truncate=False)

+--------+-------------+--------------+-----------------+-----------------+-------------------+
|explicit|nunique_pista|avg_popularity|median_popularity|avg_duration_min |median_duration_min|
+--------+-------------+--------------+-----------------+-----------------+-------------------+
|true    |17           |64.0          |70.0             |2.791194117647059|2.9761166666666665 |
+--------+-------------+--------------+-----------------+-----------------+-------------------+



In [0]:
# df_spark.filter(df_spark.release_date_bracket == "Después de 2020").show(truncate=False)

In [0]:
df_spark.filter(df_spark.release_date_bracket == "Después de 2020").groupBy('explicit').agg(countDistinct("pista").alias("nunique_pista"), \
         avg("popularity").alias("avg_popularity"), \
         median("popularity").alias("median_popularity"), \
         avg("duration_min").alias("avg_duration_min"),\
        median("duration_min").alias("median_duration_min")
     ).show(20)

+--------+-------------+------------------+-----------------+------------------+-------------------+
|explicit|nunique_pista|    avg_popularity|median_popularity|  avg_duration_min|median_duration_min|
+--------+-------------+------------------+-----------------+------------------+-------------------+
|    true|           12|59.583333333333336|             66.0|2.5326319444444443| 2.6870166666666666|
|   false|           13| 57.69230769230769|             61.0|2.2785064102564108| 2.3900333333333332|
+--------+-------------+------------------+-----------------+------------------+-------------------+

